# Example: classify time series with SOCK

This notebook is a simple example how to classify time series with SOCK features. For the purposes of this notebook, we use a simple ridge classifier on top of the SOCK features.

## 1. Make train and test paths

Let's first generate some dummy data for this notebook. Replace this with your real data.

For the dummy data, we use the TGH process from the paper. We simulate one class with tail heaviness $h=0$ and one with $h=0.15$. Each sample is an independent length-64 path.


In [1]:
import numpy as np
import torch

from sock.data.varp import VARpSimulator, tgh_transform
from sock.sock import SOCK

SEED = 0
N_STEPS = 64
torch.manual_seed(SEED)

r = omega = 0.8
eye = np.eye(3, dtype=np.float32)
simulator = VARpSimulator(
    n_steps=N_STEPS,
    lags={5: 2 * r * np.cos(omega * np.pi) * eye, 10: -r**2 * eye},
    cov=0.5 * (np.ones((3, 3), dtype=np.float32) + eye),
    rng=np.random.default_rng(SEED),
)


def make_split(n_per_class):
    paths = [
        tgh_transform(torch.from_numpy(simulator.simulate(n_per_class, burn=256)), g=0.0, h=h)
        for h in (0.0, 0.15)
    ]
    labels = torch.cat([torch.full((n_per_class,), label) for label in (0, 1)])
    return torch.cat(paths), labels


x_train, y_train = make_split(256)
x_test, y_test = make_split(128)
print(f"Train: {x_train.shape}, test: {x_test.shape}")


Train: torch.Size([512, 64, 3]), test: torch.Size([256, 64, 3])


## 2. Extract SOCK features

Fit SOCK's input and feature scales on training paths only, then apply the same fixed map to test paths.


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
sock = SOCK(
    num_channels=3,
    n_steps=N_STEPS,
    k=3,
    mix_dim=64,
    augmentations=("cumsum",),
    device=device,
)

with torch.no_grad():
    train_features = sock.forward_and_fit_scales(x_train.to(device))
    test_features = sock(x_test.to(device))


## 3. Fit a ridge classifier and report accuracy

Encode the two classes as $-1$ and $+1$, fit a linear ridge model, and threshold its score at zero.


In [3]:
target = 2 * y_train.to(device).float() - 1
alpha = 1.0
with torch.no_grad():
    weights = torch.linalg.solve(
        train_features.T @ train_features
        + alpha * torch.eye(train_features.shape[1], device=device),
        train_features.T @ target,
    )
    prediction = (test_features @ weights > 0).long()
    accuracy = (prediction == y_test.to(device)).float().mean().item()

print(f"Test accuracy: {accuracy:.1%}")


Test accuracy: 98.4%
